In [ ]:
!pip install gcloud
!gcloud auth application-default login

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 454.4/454.4 kB 7.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for gcloud: filename=gcloud-0.18.3-py3-none-any.whl size=602927 sha256=7bbc2eca7d291ed54e6db38c68a9f742707d9b4514791572a9d168babc879936
  Stored in directory: /root/.cache/pip/wheels/2a/62/75/3d74209bfebb8805823ae74afa28653aa1ea76d8b5a9d741ff
Successfully built gcloud
Go to the following link in your browser, and complete the sign-in prompts:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=https%3A%2F%2Fsdk.cloud.google.com%2Fapplicationdefaultauthcode.html&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=Ql2FsXkRK7uQ13Mzb1LQA3m9ycuvWN&prompt=consent&token_usage=remote&access_type=offline&code_chal

In [ ]:
import pandas as pd
import numpy as np
import os
import pandas_gbq
from google.cloud import bigquery
import glob
import openpyxl

# o Ano de 2024

In [ ]:
df = pd.read_excel('/content/Base_Estadic_2024.xlsx', sheet_name='Informática e comunicação')
df

,Cod UF,Sigla UF,Nome UF,PopUF,Região,Etic011,Etic012,Etic013,Etic014,Etic015,...,Etic347,Etic348,Etic35,Etic361,Etic362,Etic363,Etic364,Etic365,Etic366,Etic367
0,11,RO,RONDÔNIA,1746227,1 - Norte,Não,Não,Sim,Sim,Sim,...,Não,Não,Sim,Sim,Não,Sim,Não,Não,Não,Não
1,12,AC,ACRE,880631,1 - Norte,Não,Não,Sim,Não,Sim,...,Não,Não,Sim,Sim,Não,Não,Não,Não,Não,Não
2,13,AM,AMAZONAS,4281209,1 - Norte,Não,Não,Sim,Sim,Não,...,Não,Não,Sim,Sim,Não,Sim,Sim,Não,Não,Não
3,14,RR,RORAIMA,716793,1 - Norte,Sim,Sim,Sim,Sim,Sim,...,Não,Não,Sim,Sim,Não,Sim,Sim,Não,Não,Não
4,15,PA,PARÁ,8664306,1 - Norte,Não,Não,Não,Sim,Sim,...,Não,Não,Sim,Sim,Sim,Não,Sim,Não,Não,Não
5,16,AP,AMAPÁ,802837,1 - Norte,Sim,Sim,Sim,Não,Sim,...,Não,Não,Sim,Sim,Sim,Sim,Sim,Não,Não,Não
6,17,TO,TOCANTINS,1577342,1 - Norte,Não,Não,Sim,Não,Sim,...,Não,Não,Sim,Sim,Sim,Sim,Não,Não,Não,Não
7,21,MA,MARANHÃO,7010960,2 - Nordeste,Não,Sim,Sim,Sim,Sim,...,Não,Não,Sim,Sim,Não,Sim,Sim,Não,Não,Não
8,22,PI,PIAUÍ,3375646,2 - Nordeste,Sim,Sim,Sim,Sim,Sim,...,Não,Não,Sim,Sim,Sim,Sim,Sim,Não,Não,Não
9,23,CE,CEARÁ,9233656,2 - Nordeste,Sim,Sim,Sim,Sim,Sim,...,Não,Não,Sim,Sim,Sim,Não,Sim,Não,Não,Não


In [ ]:
df = df[['Cod UF', 'Etic12a7', 'Etic12b12', 'Etic12a4', 'Etic12a5', 'Etic12a6', 'Etic12b1']]
df

,Cod UF,Etic12a7,Etic12b12,Etic12a4,Etic12a5,Etic12a6,Etic12b1
0,11,Sim,Sim,Sim,Sim,Sim,Sim
1,12,Sim,Sim,Sim,Sim,Sim,Sim
2,13,Sim,Não,Sim,Sim,Sim,Sim
3,14,Sim,Não,Sim,Sim,Sim,Sim
4,15,Sim,Sim,Sim,Sim,Sim,Sim
5,16,Sim,Sim,Sim,Sim,Sim,Sim
6,17,Sim,Sim,Sim,Sim,Sim,Sim
7,21,Sim,Sim,Sim,Sim,Sim,Sim
8,22,Sim,Sim,Sim,Sim,Sim,Sim
9,23,Sim,Sim,Sim,Sim,Sim,Sim


In [ ]:
df = df.rename(columns={'Cod UF': 'cod_uf',
                        'Etic12a7':'concursos_publicos',
                        'Etic12b12':'pesquisa_satisfacao_servicos_estado',
                        'Etic12a4':'diario',
                        'Etic12a5':'legislacao',
                        'Etic12a6':'financas',
                        'Etic12b1':'ouvidoria_atendimento_cidadao'})

In [ ]:
cod_uf = pd.read_csv('/content/ESTADIC_2019 - Variáveis externas.csv', sep=',')[['UF','COD_UF']]

In [ ]:
x= cod_uf.pivot_table(columns=('UF','COD_UF'), aggfunc='size')


In [ ]:
cod_uf = pd.DataFrame(x).reset_index()[['UF','COD_UF']]

In [ ]:
df = df.merge(cod_uf, right_on='COD_UF',left_on='cod_uf')
df

,cod_uf,concursos_publicos,pesquisa_satisfacao_servicos_estado,diario,legislacao,financas,ouvidoria_atendimento_cidadao,UF,COD_UF
0,11,Sim,Sim,Sim,Sim,Sim,Sim,RO,11
1,12,Sim,Sim,Sim,Sim,Sim,Sim,AC,12
2,13,Sim,Não,Sim,Sim,Sim,Sim,AM,13
3,14,Sim,Não,Sim,Sim,Sim,Sim,RR,14
4,15,Sim,Sim,Sim,Sim,Sim,Sim,PA,15
5,16,Sim,Sim,Sim,Sim,Sim,Sim,AP,16
6,17,Sim,Sim,Sim,Sim,Sim,Sim,TO,17
7,21,Sim,Sim,Sim,Sim,Sim,Sim,MA,21
8,22,Sim,Sim,Sim,Sim,Sim,Sim,PI,22
9,23,Sim,Sim,Sim,Sim,Sim,Sim,CE,23


In [ ]:
df = df.drop(['COD_UF'], axis=1)
df

,cod_uf,concursos_publicos,pesquisa_satisfacao_servicos_estado,diario,legislacao,financas,ouvidoria_atendimento_cidadao,UF
0,11,Sim,Sim,Sim,Sim,Sim,Sim,RO
1,12,Sim,Sim,Sim,Sim,Sim,Sim,AC
2,13,Sim,Não,Sim,Sim,Sim,Sim,AM
3,14,Sim,Não,Sim,Sim,Sim,Sim,RR
4,15,Sim,Sim,Sim,Sim,Sim,Sim,PA
5,16,Sim,Sim,Sim,Sim,Sim,Sim,AP
6,17,Sim,Sim,Sim,Sim,Sim,Sim,TO
7,21,Sim,Sim,Sim,Sim,Sim,Sim,MA
8,22,Sim,Sim,Sim,Sim,Sim,Sim,PI
9,23,Sim,Sim,Sim,Sim,Sim,Sim,CE


In [ ]:
df['ano']=2024

In [ ]:
df = df.rename(columns={'UF':'sigla_uf'})


In [ ]:
df = df[['ano','sigla_uf','cod_uf', 'concursos_publicos', 'pesquisa_satisfacao_servicos_estado', 'ouvidoria_atendimento_cidadao', 'diario', 'legislacao', 'financas']]

In [ ]:
df

,ano,sigla_uf,cod_uf,concursos_publicos,pesquisa_satisfacao_servicos_estado,ouvidoria_atendimento_cidadao,diario,legislacao,financas
0,2024,RO,11,Sim,Sim,Sim,Sim,Sim,Sim
1,2024,AC,12,Sim,Sim,Sim,Sim,Sim,Sim
2,2024,AM,13,Sim,Não,Sim,Sim,Sim,Sim
3,2024,RR,14,Sim,Não,Sim,Sim,Sim,Sim
4,2024,PA,15,Sim,Sim,Sim,Sim,Sim,Sim
5,2024,AP,16,Sim,Sim,Sim,Sim,Sim,Sim
6,2024,TO,17,Sim,Sim,Sim,Sim,Sim,Sim
7,2024,MA,21,Sim,Sim,Sim,Sim,Sim,Sim
8,2024,PI,22,Sim,Sim,Sim,Sim,Sim,Sim
9,2024,CE,23,Sim,Sim,Sim,Sim,Sim,Sim


In [ ]:
df.columns

Index(['ano', 'sigla_uf', 'cod_uf', 'concursos_publicos',
       'pesquisa_satisfacao_servicos_estado', 'ouvidoria_atendimento_cidadao',
       'diario', 'legislacao', 'financas'],
      dtype='object')

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27 entries, 0 to 26
Data columns (total 9 columns):
 #   Column                               Non-Null Count  Dtype 
---  ------                               --------------  ----- 
 0   ano                                  27 non-null     int64 
 1   sigla_uf                             27 non-null     object
 2   cod_uf                               27 non-null     int64 
 3   concursos_publicos                   27 non-null     object
 4   pesquisa_satisfacao_servicos_estado  27 non-null     object
 5   ouvidoria_atendimento_cidadao        27 non-null     object
 6   diario                               27 non-null     object
 7   legislacao                           27 non-null     object
 8   financas                             27 non-null     object
dtypes: int64(2), object(7)
memory usage: 2.0+ KB


# Consumindo o ano de 2019 através do GBQ

In [ ]:


query = """SELECT * FROM `repositoriodedadosgpsp.participacao_transparencia.ESTADIC_transparencia_internet` WHERE ano = 2019"""
# Execute the query using pandas_gbq.read_gbq and load the result into a pandas DataFrame called 'df'.
# The 'project_id' specifies the Google Cloud Project to use.
df_2019 = pandas_gbq.read_gbq(query, project_id='repositoriodedadosgpsp')



Downloading: 100%|██████████|


In [ ]:
df_2019.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27 entries, 0 to 26
Data columns (total 7 columns):
 #   Column                               Non-Null Count  Dtype 
---  ------                               --------------  ----- 
 0   ano                                  27 non-null     Int64 
 1   sigla_uf                             27 non-null     object
 2   cod_uf                               27 non-null     Int64 
 3   concursos_publicos                   27 non-null     object
 4   pesquisa_satisfacao_servicos_estado  27 non-null     object
 5   diario_legislacao_financas           27 non-null     object
 6   ouvidoria_atendimento_cidadao        27 non-null     object
dtypes: Int64(2), object(5)
memory usage: 1.7+ KB


In [ ]:
for col in ['diario', 'legislacao', 'financas']:
    df_2019[col] = df_2019['diario_legislacao_financas']

In [ ]:
df_2019.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27 entries, 0 to 26
Data columns (total 10 columns):
 #   Column                               Non-Null Count  Dtype 
---  ------                               --------------  ----- 
 0   ano                                  27 non-null     Int64 
 1   sigla_uf                             27 non-null     object
 2   cod_uf                               27 non-null     Int64 
 3   concursos_publicos                   27 non-null     object
 4   pesquisa_satisfacao_servicos_estado  27 non-null     object
 5   diario_legislacao_financas           27 non-null     object
 6   ouvidoria_atendimento_cidadao        27 non-null     object
 7   diario                               27 non-null     object
 8   legislacao                           27 non-null     object
 9   financas                             27 non-null     object
dtypes: Int64(2), object(8)
memory usage: 2.3+ KB


In [ ]:
df_2019.head(10)

,ano,sigla_uf,cod_uf,concursos_publicos,pesquisa_satisfacao_servicos_estado,diario_legislacao_financas,ouvidoria_atendimento_cidadao,diario,legislacao,financas
0,2019,RO,11,Sim,Sim,Sim,Sim,Sim,Sim,Sim
1,2019,AP,16,Sim,Sim,Sim,Sim,Sim,Sim,Sim
2,2019,TO,17,Sim,Sim,Sim,Sim,Sim,Sim,Sim
3,2019,CE,23,Sim,Sim,Sim,Sim,Sim,Sim,Sim
4,2019,PB,25,Sim,Sim,Sim,Sim,Sim,Sim,Sim
5,2019,SE,28,Sim,Sim,Sim,Sim,Sim,Sim,Sim
6,2019,MG,31,Sim,Sim,Sim,Sim,Sim,Sim,Sim
7,2019,ES,32,Sim,Sim,Sim,Sim,Sim,Sim,Sim
8,2019,MT,51,Sim,Sim,Sim,Não,Sim,Sim,Sim
9,2019,AC,12,Sim,Não,Sim,Sim,Sim,Sim,Sim


In [ ]:
df_2019 = df_2019.drop(columns=['diario_legislacao_financas'])

In [ ]:
df_2019.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27 entries, 0 to 26
Data columns (total 9 columns):
 #   Column                               Non-Null Count  Dtype 
---  ------                               --------------  ----- 
 0   ano                                  27 non-null     Int64 
 1   sigla_uf                             27 non-null     object
 2   cod_uf                               27 non-null     Int64 
 3   concursos_publicos                   27 non-null     object
 4   pesquisa_satisfacao_servicos_estado  27 non-null     object
 5   ouvidoria_atendimento_cidadao        27 non-null     object
 6   diario                               27 non-null     object
 7   legislacao                           27 non-null     object
 8   financas                             27 non-null     object
dtypes: Int64(2), object(7)
memory usage: 2.1+ KB


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27 entries, 0 to 26
Data columns (total 9 columns):
 #   Column                               Non-Null Count  Dtype 
---  ------                               --------------  ----- 
 0   ano                                  27 non-null     int64 
 1   sigla_uf                             27 non-null     object
 2   cod_uf                               27 non-null     int64 
 3   concursos_publicos                   27 non-null     object
 4   pesquisa_satisfacao_servicos_estado  27 non-null     object
 5   ouvidoria_atendimento_cidadao        27 non-null     object
 6   diario                               27 non-null     object
 7   legislacao                           27 non-null     object
 8   financas                             27 non-null     object
dtypes: int64(2), object(7)
memory usage: 2.0+ KB


In [ ]:
df_final = pd.concat([df, df_2019], ignore_index=True)

In [ ]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54 entries, 0 to 53
Data columns (total 9 columns):
 #   Column                               Non-Null Count  Dtype 
---  ------                               --------------  ----- 
 0   ano                                  54 non-null     Int64 
 1   sigla_uf                             54 non-null     object
 2   cod_uf                               54 non-null     Int64 
 3   concursos_publicos                   54 non-null     object
 4   pesquisa_satisfacao_servicos_estado  54 non-null     object
 5   ouvidoria_atendimento_cidadao        54 non-null     object
 6   diario                               54 non-null     object
 7   legislacao                           54 non-null     object
 8   financas                             54 non-null     object
dtypes: Int64(2), object(7)
memory usage: 4.0+ KB


# Subindo para o GBQ

In [ ]:
# Define the BigQuery table schema with Portuguese descriptions
schema=[bigquery.SchemaField('ano','INTEGER',description='Ano de referência da observação'),
        bigquery.SchemaField('cod_uf','INTEGER',description='Código do IBGE da UF'),
        bigquery.SchemaField('sigla_uf','STRING',description='Sigla da Unidade da Federação'),
        bigquery.SchemaField('concursos_publicos','STRING',description='Serviços disponibilizados na internet sobre concursos públicos'),
        bigquery.SchemaField('pesquisa_satisfacao_servicos_estado','STRING',description='Serviços disponibilizados na internet sobre pesquisa de satisfação relacionada aos serviços prestados pelo estado'),
        bigquery.SchemaField('diario','STRING',description='Serviços disponibilizados na internet sobre diário oficial'),
        bigquery.SchemaField('legislacao','STRING',description='Serviços disponibilizados na internet sobre legislação estadual e finanças públicas'),
        bigquery.SchemaField('financas','STRING',description='Serviços disponibilizados na internet sobre finanças públicas'),
        bigquery.SchemaField('ouvidoria_atendimento_cidadao','STRING',description='Serviços disponibilizados na internet sobre ouvidoria e serviços de atendimento ao cidadão'),
]

# Initialize BigQuery client connection
client = bigquery.Client(project='repositoriodedadosgpsp')

# Create reference to target dataset
dataset_ref = client.dataset('participacao_transparencia')

# Create reference to target table with standardized naming convention:
# FONTE_algo_intuitivo_dado (MUNIC_quantidade_vinculos_mapa_v1)
table_ref = dataset_ref.table('ESTADIC_transparencia_internet_v1')

# Configure the load job with our schema definition
job_config = bigquery.LoadJobConfig(
    schema=schema,
    # Optional parameters (commented out):
    # write_disposition="WRITE_TRUNCATE",  # Overwrites table if exists
    # create_disposition="CREATE_IF_NEEDED"  # Default behavior
)

# Execute the load job to upload DataFrame to BigQuery
job = client.load_table_from_dataframe(
    dataframe=df_final,
    destination=table_ref,
    job_config=job_config
)

# Wait for the job to complete
job.result()

/usr/local/lib/python3.12/dist-packages/google/auth/_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


LoadJob<project=repositoriodedadosgpsp, location=US, id=dc3decb0-0998-443e-a1d1-303697a9bdc7>